# 05b — Classification of one-road starting advantage
Run 05_starting_advantage first. This optional notebook predicts the theoretical benchmark, not actual winners. Blocked players are identified geometrically and excluded from the supervised fit. Training-only quartiles define Strong, Average and Weak; game-grouped validation prevents sharing boards across splits.

In [ ]:
from pathlib import Path
import pandas as pd
from advantage_analysis import compare, FEATURES
from experiment_io import OUT, read_required, validate_advantage, same_players, validate_players
print("Working directory:",Path.cwd())
d=validate_advantage(read_required(OUT,"starting_advantage.csv","05_starting_advantage.ipynb"))
p=validate_players(read_required(OUT,"initial_placements.csv","01_data_extraction.ipynb"))
assert same_players(d,p), "Starting advantage is from a different dataset. Rerun 05a."
print(f"Games: {d.Game_ID.nunique():,}; players: {len(d):,}; eligible for classification: {(~d.Blocked).sum():,}")
display(d.Status.value_counts())

In [ ]:
# Each classifier is evaluated on a grouped holdout and five grouped CV folds.
results, cv, predictions, labeled, q = compare(d)
assert q[1]<40, "Horizon must exceed the weak-class boundary"
print("Train-only quartile boundaries:",q)
print("Features:",FEATURES)
display(labeled.groupby(["Split","Advantage_Class"]).size())
display(results.round(4))
display(cv.round(4))
results.to_csv(OUT/"advantage_classification_results.csv",index=False)
cv.to_csv(OUT/"advantage_classification_cv.csv",index=False)
predictions.to_csv(OUT/"advantage_predictions.csv",index=False)
labeled.to_csv(OUT/"advantage_labeled_positions.csv",index=False)
print("Classification files saved to",OUT.resolve())

The outputs are newly calculated from the 5,000-game dataset. Use 06 for comparison charts and 07/08 for actual winner analysis.